# Training Data Analysis

This notebook computes summary statistics for the training data analysis (Section 6 of the paper).
It reports the average masculine prevalence in the training corpus and the breakdown of examples
by predicted gender and training prevalence direction (Table 1 / Table 7).

## Inputs

| Name | Description | How to obtain |
|---|---|---|
| `summary_dataframe.tsv` | One row per gender term instance, with model predictions, ILM probabilities, flip thresholds, and training data term frequencies. | Output of `prepare_df_for_analysis_clean.ipynb` (`output_path`). |

## Output

Printed statistics corresponding to Section 6 of the paper:
- Mean ± std of the masculine training prevalence across all filtered examples
- Count table: predicted gender (F / M) × whether the predicted form is more frequent in training data (True / False)

## Imports

In [1]:
import csv

import numpy as np
import pandas as pd

## Configuration

Set all paths and parameters in this cell before running the notebook.

In [2]:
LANG = "it"            # one of: es, fr, it
MODEL = "conformer"    # one of: transformer, conformer

# summary_dataframe.tsv – output of prepare_df_for_analysis_clean.ipynb (output_path)
SUMMARY_TSV = f"/path/to/results/{MODEL}/en-{LANG}/summary_dataframe.tsv"

## 1. Load and Filter Data

Load the summary dataframe and restrict to the subset used in Section 6:
category-1 terms (speaker-referring) with articles and prepositions excluded,
following Savoldi et al. (2022b).

In [4]:
df = pd.read_csv(SUMMARY_TSV, sep='\t', escapechar='\\', quoting=csv.QUOTE_NONE)

# Category 1 = speaker-referring terms; Art/Prep excluded following Savoldi et al. (2022)
filtered_df = df[(df.cat_nb == 1) & (df.pos != 'Art/Prep')].copy()

# Predicted gender: reference gender when correct, opposite when incorrect
filtered_df['predicted_gender'] = filtered_df.apply(
    lambda x: x.cat_gender if x.correct else ('F' if x.cat_gender == 'M' else 'M'),
    axis=1,
)

# Masculine training prevalence: fraction of masculine occurrences for this term pair.
# training_pref_for_predicted gives the prevalence of whichever form was generated;
# we flip it for feminine predictions to always express the masculine fraction.
filtered_df['training_pref_for_masc'] = filtered_df.apply(
    lambda x: x['training_pref_for_predicted'] if x['predicted_gender'] == 'M'
              else 1 - x['training_pref_for_predicted'],
    axis=1,
)

# True if the predicted form is more frequent than its alternative in training data
filtered_df['predicted_more_frequent'] = filtered_df['training_pref_for_predicted'] > 0.5

## 2. Average Masculine Prevalence in Training Data

Corresponds to the per-language masculine skew values reported in Section 6.

In [5]:
filtered_df['training_pref_for_masc'].mean(), filtered_df['training_pref_for_masc'].std()

(0.6840907099594433, 0.1545927639139493)

## 3. Predicted Gender vs. Training Prevalence

Count of examples by predicted gender and whether the predicted form is more or less
frequent in training data. Corresponds to Table 1 (Transformer) or Table 7 (Conformer).

In [6]:
filtered_df.groupby(['predicted_gender', 'predicted_more_frequent'])['id'].count() \
    .unstack() \
    .reindex([True, False], axis=1) \
    .rename(columns={True: 'More Freq.', False: 'Less Freq.'})

predicted_more_frequent,More Freq.,Less Freq.
predicted_gender,,
F,13,105
M,179,12
